In [ ]:
!pip install groq tavily-python beautifulsoup4 httpx -q

In [ ]:
import os
from google.colab import userdata


os.environ["groq_api"] = userdata.get("groq_API")
os.environ["TAVILY_API_KEY"]    = userdata.get("TAVILY_API_KEY")

print("✅ API keys loaded")

In [ ]:
import httpx
from bs4 import BeautifulSoup

TAVILY_KEY = os.environ["TAVILY_API_KEY"]

def search_web(query: str) -> list:
    resp = httpx.post(
        "https://api.tavily.com/search",
        json={"query": query, "api_key": TAVILY_KEY, "max_results": 5},
        timeout=15
    )
    results = resp.json().get("results", [])
    print(f"    🔍 Searched: '{query}' → {len(results)} results")
    return [{"title": r["title"], "url": r["url"], "snippet": r.get("content", "")[:300]} for r in results]

def scrape_page(url: str) -> str:
    try:
        html = httpx.get(url, timeout=10, follow_redirects=True).text
        soup = BeautifulSoup(html, "html.parser")
        for tag in soup(["script", "style", "nav", "footer"]):
            tag.decompose()
        text = soup.get_text(separator="\n", strip=True)
        print(f"    🌐 Scraped: {url[:60]}...")
        return text[:5000]
    except Exception as e:
        return f"Error scraping {url}: {e}"

visited_urls = set()

def save_report(filename: str, content: str) -> str:
    if not filename.endswith(".md"):
        filename += ".md"
    with open(filename, "w") as f:
        f.write(content)
    print(f"    💾 Report saved: {filename}")
    return f"Report saved to {filename}"

print("✅ Tools defined")

In [ ]:
import json
from groq import Groq

client = Groq(api_key=os.environ["groq_api"])

TOOLS = [
    {
        "type": "function",
        "function": {
            "name": "search_web",
            "description": "Search the internet for information on a topic. Returns titles, URLs, and snippets.",
            "parameters": {
                "type": "object",
                "properties": {
                    "query": {"type": "string", "description": "The search query to use"}
                },
                "required": ["query"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "scrape_page",
            "description": "Read the full text content of a web page given its URL.",
            "parameters": {
                "type": "object",
                "properties": {
                    "url": {"type": "string", "description": "The full URL to scrape"}
                },
                "required": ["url"]
            }
        }
    },
    {
        "type": "function",
        "function": {
            "name": "save_report",
            "description": "Save the completed research report as a Markdown file.",
            "parameters": {
                "type": "object",
                "properties": {
                    "filename": {"type": "string", "description": "Output filename (e.g. report.md)"},
                    "content":  {"type": "string", "description": "The full Markdown report content"}
                },
                "required": ["filename", "content"]
            }
        }
    }
]

TOOL_MAP = {
    "search_web":  search_web,
    "scrape_page": scrape_page,
    "save_report": save_report,
}

SYSTEM = """You are a thorough research assistant. When given a topic:
1. Run at least 3 different web searches to gather broad coverage.
2. Scrape 2–3 of the most relevant URLs for deeper detail.
3. Synthesize everything into a well-structured Markdown report with:
   - A title and introduction
   - Key findings (with sub-sections)
   - A conclusion
   - A sources section listing all URLs you used
4. Save the report using save_report with a descriptive filename.
Be thorough. Do not stop until the report is complete and saved."""

def run_research_agent(topic: str):
    print(f"\n🤖 Starting research on: '{topic}'\n{'─'*50}")
    visited_urls.clear()

    messages = [
        {"role": "system", "content": SYSTEM},
        {"role": "user",   "content": f"Research this topic and save a comprehensive report: {topic}"}
    ]
    step = 0

    while True:
        step += 1
        print(f"\n📋 Step {step} — thinking...")

        response = client.chat.completions.create(
            model="llama-3.3-70b-versatile",   # best Groq model for tool use
            messages=messages,
            tools=TOOLS,
            tool_choice="auto",
            max_tokens=4096,
            temperature=0.3
        )

        msg = response.choices[0].message

        messages.append({
            "role": "assistant",
            "content": msg.content or "",
            "tool_calls": [
                {
                    "id": tc.id,
                    "type": "function",
                    "function": {"name": tc.function.name, "arguments": tc.function.arguments}
                }
                for tc in (msg.tool_calls or [])
            ] or None
        })

        if msg.content and msg.content.strip():
            print(f"\n💭 Agent: {msg.content[:200]}{'...' if len(msg.content) > 200 else ''}")

        if not msg.tool_calls:
            print(f"\n{'─'*50}\n✅ Research complete! Check the Colab file browser for your report.")
            break

        for tc in msg.tool_calls:
            name = tc.function.name
            args = json.loads(tc.function.arguments)
            print(f"\n  🔧 Tool: {name}")

            result = TOOL_MAP[name](**args)

            messages.append({
                "role": "tool",
                "tool_call_id": tc.id,
                "content": json.dumps(result) if isinstance(result, list) else str(result)
            })

print("✅ Agent loop ready")

In [ ]:
TOPIC = "The impact of large language models on scientific research"

run_research_agent(TOPIC)

In [ ]:
import glob

reports = sorted(glob.glob("*.md"), key=lambda f: __import__("os").path.getmtime(f), reverse=True)

if reports:
    latest = reports[0]
    print(f"📄 Displaying: {latest}\n{'═'*60}\n")
    with open(latest) as f:
        print(f.read())
else:
    print("No report file found yet — make sure Cell 5 ran successfully.")

In [ ]:
!pip install langchain langchain-groq langchain-community tavily-python beautifulsoup4 httpx -q

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"]        = userdata.get("groq_API")
os.environ["TAVILY_API_KEY"]      = userdata.get("TAVILY_API_KEY")
os.environ["LANGCHAIN_TRACING_V2"] = "false"

print("✅ Keys loaded")

In [ ]:
from langchain.tools import tool
from tavily import TavilyClient
import httpx
from bs4 import BeautifulSoup

tavily = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@tool
def search_web(query: str) -> str:
    """Search the web for information on a topic."""
    results = tavily.search(query=query, max_results=4)["results"]
    return "\n".join([f"- {r['title']}: {r['content'][:200]}" for r in results])

@tool
def scrape_url(url: str) -> str:
    """Scrape and return the text content of a webpage."""
    try:
        html = httpx.get(url, timeout=10, follow_redirects=True).text
        soup = BeautifulSoup(html, "html.parser")
        for tag in soup(["script", "style", "nav"]):
            tag.decompose()
        return soup.get_text(separator="\n", strip=True)[:4000]
    except Exception as e:
        return f"Error: {e}"

tools = [search_web, scrape_url]
print("✅ Tools ready:", [t.name for t in tools])

In [ ]:
from langchain_groq import ChatGroq
from langchain.agents import AgentExecutor, create_tool_calling_agent
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.messages import HumanMessage, AIMessage

llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2)

researcher_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a research specialist. Your job is to:
1. Search the web thoroughly (at least 3 searches)
2. Scrape 2-3 relevant pages for deeper content
3. Return a structured set of findings with sources

Always cite your sources. Be factual and comprehensive."""),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

researcher_agent = create_tool_calling_agent(llm, tools, researcher_prompt)
researcher = AgentExecutor(agent=researcher_agent, tools=tools, verbose=True, max_iterations=8)

writer_prompt = ChatPromptTemplate.from_messages([
    ("system", """You are a professional report writer. Given research findings:
1. Write a clear, well-structured Markdown report
2. Include: Title, Introduction, Key Findings (with sections), Conclusion, Sources
3. Make it readable and insightful — not just a list of facts
4. Length: 400-600 words"""),
    MessagesPlaceholder("chat_history", optional=True),
    ("human", "{input}"),
    MessagesPlaceholder("agent_scratchpad"),
])

writer_agent = create_tool_calling_agent(llm, [], writer_prompt)
writer = AgentExecutor(agent=writer_agent, tools=[], verbose=True)

print("✅ Researcher and Writer agents ready")

In [ ]:
!pip install -q \
  requests==2.32.4 \
  langchain \
  langchain-groq \
  langchain-community==0.3.24 \
  langgraph \
  tavily-python \
  beautifulsoup4 \
  httpx

# Verify
import importlib.metadata
print("requests:            ", importlib.metadata.version("requests"))
print("langchain:           ", importlib.metadata.version("langchain"))
print("langchain-community: ", importlib.metadata.version("langchain-community"))
print("langgraph:           ", importlib.metadata.version("langgraph"))
print("✅ Done — restart runtime now (Runtime → Restart session)")

In [ ]:
# !pip install -q requests==2.32.4
# !pip install -q langchain langchain-groq langchain-community==0.3.24 langgraph tavily-python beautifulsoup4 httpx

import importlib.metadata
print("langchain :", importlib.metadata.version("langchain"))
print("langgraph :", importlib.metadata.version("langgraph"))
print("requests  :", importlib.metadata.version("requests"))
print("✅ Done — restart runtime now")

In [ ]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"]   = userdata.get("groq")
os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
print("✅ Keys loaded")

In [ ]:
from typing import TypedDict
from langchain_groq import ChatGroq
from langchain_core.tools import tool
from langchain_core.messages import SystemMessage, HumanMessage, ToolMessage
from tavily import TavilyClient
import httpx, json
from bs4 import BeautifulSoup

# ── Shared state ──────────────────────────────────────────────────
class State(TypedDict):
    topic:            str
    research_findings: str
    human_feedback:   str
    reject_reason:    str
    final_report:     str
    iterations:       int

# ── LLM ──────────────────────────────────────────────────────────
llm = ChatGroq(model="llama-3.1-8b-instant", temperature=0.2)

# ── Tools ─────────────────────────────────────────────────────────
tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

@tool
def search_web(query: str) -> str:
    """Search the web for information on a topic."""
    results = tavily_client.search(query=query, max_results=4)["results"]
    return "\n".join([f"- {r['title']}: {r['content'][:250]}" for r in results])

@tool
def scrape_url(url: str) -> str:
    """Scrape full text from a webpage URL."""
    try:
        html = httpx.get(url, timeout=10, follow_redirects=True).text
        soup = BeautifulSoup(html, "html.parser")
        for tag in soup(["script","style","nav","footer"]):
            tag.decompose()
        return soup.get_text(separator="\n", strip=True)[:4000]
    except Exception as e:
        return f"Error: {e}"

tools    = [search_web, scrape_url]
tool_map = {t.name: t for t in tools}

print("✅ State + tools ready")

In [ ]:
# ── Helper: run one LLM tool-calling loop ─────────────────────────
def run_llm_with_tools(system: str, user_input: str, max_steps: int = 8) -> str:
    bound = llm.bind_tools(tools)
    messages = [SystemMessage(content=system), HumanMessage(content=user_input)]

    for _ in range(max_steps):
        response = bound.invoke(messages)
        messages.append(response)

        if not getattr(response, "tool_calls", None):
            return response.content

        for tc in response.tool_calls:
            print(f"    🔧 {tc['name']}({str(list(tc['args'].values()))[:60]})")
            result = tool_map[tc["name"]].invoke(tc["args"])
            messages.append(ToolMessage(content=str(result), tool_call_id=tc["id"]))

    return messages[-1].content

# ── Node 1: Researcher ────────────────────────────────────────────
def researcher_node(state: State) -> State:
    iteration = state.get("iterations", 0) + 1
    reject_reason = state.get("reject_reason", "")

    print(f"\n📚 RESEARCHER (run #{iteration})")

    extra = ""
    if reject_reason:
        extra = f"\n\nPrevious attempt was REJECTED with feedback: '{reject_reason}'. Please fix this."

    findings = run_llm_with_tools(
        system="""You are a research specialist. You MUST:
1. Run at least 3 different web searches on different angles of the topic
2. Scrape at least 2 relevant URLs for depth
3. Return a thorough, structured summary with all sources listed
Be comprehensive. Include facts, statistics, and expert perspectives.""",
        user_input=f"Research this topic thoroughly: {state['topic']}{extra}"
    )

    print(f"  ✅ Research done ({len(findings)} chars)")
    return {**state, "research_findings": findings, "human_feedback": "", "iterations": iteration}

# ── Node 2: Human approval (the PAUSE point) ─────────────────────
def human_approval_node(state: State) -> State:
    print("\n" + "="*55)
    print("  ⏸  GRAPH PAUSED — HUMAN REVIEW REQUIRED")
    print("="*55)
    print(f"\nTopic: {state['topic']}")
    print(f"Iteration: #{state['iterations']}")
    print("\n--- RESEARCH FINDINGS PREVIEW ---")
    print(state["research_findings"][:1200])
    print("..." if len(state["research_findings"]) > 1200 else "")
    print("\n" + "-"*55)

    while True:
        decision = input("\n✋ Your decision — type 'approve' or 'reject': ").strip().lower()
        if decision == "approve":
            print("  ✅ Approved! Continuing to writer...")
            return {**state, "human_feedback": "approved", "reject_reason": ""}
        elif decision == "reject":
            reason = input("  📝 Reason for rejection (what should be improved?): ").strip()
            print("  🔄 Sending back to researcher with your feedback...")
            return {**state, "human_feedback": "rejected", "reject_reason": reason}
        else:
            print("  Please type exactly 'approve' or 'reject'.")

# ── Node 3: Writer ────────────────────────────────────────────────
def writer_node(state: State) -> State:
    print("\n✍️  WRITER: Composing report...")

    report = llm.invoke(f"""Write a professional, well-structured Markdown research report.

TOPIC: {state['topic']}

RESEARCH FINDINGS:
{state['research_findings']}

Structure:
# [Title]
## Introduction
## Key Findings
### [Sub-section per major theme]
## Conclusion
## Sources

Target: 500-700 words. Be insightful and analytical, not just a list.""").content

    print(f"  ✅ Report written ({len(report)} chars)")
    return {**state, "final_report": report}

# ── Node 4: Save report ───────────────────────────────────────────
def save_node(state: State) -> State:
    filename = state["topic"].replace(" ", "_")[:40].lower() + "_report.md"
    with open(filename, "w") as f:
        f.write(state["final_report"])
    print(f"\n💾 Report saved: {filename}")
    return state

# ── Routing function ──────────────────────────────────────────────
def route_after_human(state: State) -> str:
    if state["human_feedback"] == "approved":
        return "writer"
    else:
        return "researcher"

print("✅ All nodes defined")

In [ ]:
from langgraph.graph import StateGraph, END

graph = StateGraph(State)

# Add nodes
graph.add_node("researcher",     researcher_node)
graph.add_node("human_approval", human_approval_node)
graph.add_node("writer",         writer_node)
graph.add_node("save",           save_node)

# Entry point
graph.set_entry_point("researcher")

# Edges
graph.add_edge("researcher",     "human_approval")

graph.add_conditional_edges(
    "human_approval",
    route_after_human,
    {
        "writer":     "writer",
        "researcher": "researcher",
    }
)

graph.add_edge("writer", "save")
graph.add_edge("save",   END)

app = graph.compile()
print("✅ Graph compiled")

In [ ]:
TOPIC = "The impact of AI agents on software engineering jobs in 2026"

initial_state: State = {
    "topic":             TOPIC,
    "research_findings": "",
    "human_feedback":    "",
    "reject_reason":     "",
    "final_report":      "",
    "iterations":        0,
}

print(f"\n🚀 Starting HITL research agent for: '{TOPIC}'")
print("   You will be asked to approve or reject the research before the report is written.\n")

final = app.invoke(initial_state)

print("\n" + "="*55)
print("FINAL REPORT:")
print("="*55)
print(final["final_report"][:800] + "...")